# 01 - Data Cleaning & Customer Analytics

## 1. Business Understanding

**Business:** UK-based online gift retailer (Dec 2010 - Dec 2011). Many customers are likely wholesalers (to be verified from the data).

**Goal:** Understand customer behavior and build actionable segments to decide who to reward, who to win back, and who to nudge toward a repeat purchase.

**Business questions:**
1. Which products generate the most units and revenue?
2. Which customers contribute the most revenue and orders?
3. How does revenue change over time?
4. Which months, weekdays, and hours are strongest?
5. Which countries generate the most customers, orders, and revenue?
6. Are there seasonal purchasing patterns?
7. How common are cancellations and returns, and which products/customers are associated with them?
8. What does typical customer purchasing behavior look like?
9. Are there signs of wholesale/bulk buying?
10. Can customers be grouped into actionable segments based on value and engagement?

## 2. Dataset Loading & Documentation

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/Online Retail.xlsx")

In [2]:
print("Shape:", df.shape)
df.head()

Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


### Column Description

| Column | Description | Dtype (as loaded) |
|---|---|---|
| InvoiceNo | Invoice number. Starts with `C` for cancellations/returns | object |
| StockCode | Product code | object |
| Description | Product name | object |
| Quantity | Units per invoice line (negative = return) | int64 |
| InvoiceDate | Invoice date and time | datetime64 |
| UnitPrice | Price per unit (GBP) | float64 |
| CustomerID | Customer identifier (missing for guest checkouts) | float64 |
| Country | Customer country | object |

**Observations:**
- Each row is an invoice **line**, not a full order.
- `CustomerID` was loaded as float64 because of missing values. It is an identifier, so it will be converted to string in Cleaning.
- `CustomerID` is missing in about 25% of rows. `Description` is missing in 1,454 rows. Both will be investigated in the Data Audit.

## 3. Data Audit

Inspection only. No rows are modified or removed in this section.

In [4]:
missing = df.isna().sum().to_frame("missing")
missing["pct"] = (missing["missing"] / len(df) * 100).round(2)
missing

,missing,pct
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135080,24.93
Country,0,0.00


In [5]:
print("Fully duplicated rows:", df.duplicated().sum())

Fully duplicated rows: 5268


In [6]:
df[["Quantity", "UnitPrice"]].describe()

,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


In [7]:
is_cancel = df["InvoiceNo"].astype(str).str.startswith("C")

print("Cancellation lines (InvoiceNo starts with C):", is_cancel.sum())
print("Negative Quantity lines:", (df["Quantity"] < 0).sum())
print("Negative Quantity but NOT a cancellation:", ((df["Quantity"] < 0) & ~is_cancel).sum())
print("Cancellation with positive Quantity:", (is_cancel & (df["Quantity"] > 0)).sum())
print("UnitPrice == 0:", (df["UnitPrice"] == 0).sum())
print("UnitPrice < 0:", (df["UnitPrice"] < 0).sum())

Cancellation lines (InvoiceNo starts with C): 9288
Negative Quantity lines: 10624
Negative Quantity but NOT a cancellation: 1336
Cancellation with positive Quantity: 0
UnitPrice == 0: 2515
UnitPrice < 0: 2


In [8]:
no_cust = df["CustomerID"].isna()

print("Rows without CustomerID:", no_cust.sum())
print("Unique invoices without CustomerID:", df.loc[no_cust, "InvoiceNo"].nunique())

# Do any invoices mix rows with and without CustomerID?
mixed = df.groupby("InvoiceNo")["CustomerID"].apply(lambda s: s.isna().any() and s.notna().any())
print("Invoices with mixed missing/non-missing CustomerID:", mixed.sum())

print("\nCountry split for rows without CustomerID:")
print(df.loc[no_cust, "Country"].value_counts().head())

Rows without CustomerID: 135080
Unique invoices without CustomerID: 3710
Invoices with mixed missing/non-missing CustomerID: 0

Country split for rows without CustomerID:
Country
United Kingdom    133600
EIRE                 711
Hong Kong            288
Unspecified          202
Switzerland          125
Name: count, dtype: int64


In [9]:
print("Missing Description rows with UnitPrice == 0:",
      (df["Description"].isna() & (df["UnitPrice"] == 0)).sum())
print("Missing Description rows total:", df["Description"].isna().sum())

# StockCodes that are not regular product codes (e.g. POST, D, M, BANK CHARGES)
non_product = df.loc[~df["StockCode"].astype(str).str.match(r"^\d{5}"), "StockCode"]
print("\nNon-standard StockCodes:")
print(non_product.value_counts().head(15))

print("\nDate range:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
print("Countries:", df["Country"].nunique())

Missing Description rows with UnitPrice == 0: 1454
Missing Description rows total: 1454

Non-standard StockCodes:
StockCode
POST            1256
DOT              710
M                571
C2               144
D                 77
S                 63
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
DCGSSBOY          11
gift_0001_20      10
gift_0001_10       9
gift_0001_30       8
DCGS0003           5
Name: count, dtype: int64

Date range: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00
Countries: 38


### Audit Findings

| Issue | Count | Implication |
|---|---|---|
| Missing CustomerID | 135,080 rows (24.9%), 3,710 invoices | Whole invoices are guest checkouts (no mixed invoices). ~99% are UK. Cannot be used for customer-level analysis |
| Missing Description | 1,454 rows | All have UnitPrice = 0: system adjustments, not sales |
| Fully duplicated rows | 5,268 (~1%) | Likely entry duplicates. Decide in Cleaning |
| Cancellation lines (C prefix) | 9,288 | All have negative Quantity. Need netting against original invoices |
| Negative Quantity, not a cancellation | 1,336 | Likely adjustments/damaged stock. Investigate |
| UnitPrice = 0 | 2,515 | Giveaways, samples, or system errors |
| UnitPrice < 0 | 2 | Adjustments (e.g. bad debt) |
| Extreme values | Quantity ±80,995, UnitPrice up to 38,970 | Outliers. Same magnitude positive/negative suggests a cancelled order |
| Non-product StockCodes | POST, DOT, M, BANK CHARGES, AMAZONFEE, CRUK, gift_* | Fees and services, not products. Exclude from product analysis |
| Skewed distributions | Mean Quantity 9.55 vs median 3 | Log transform needed before clustering |
| Date range | 2010-12-01 to 2011-12-09 | December 2011 is incomplete. Affects seasonality interpretation |

**Decisions to take in Cleaning:** handle duplicates, isolate guest checkouts, separate cancellations, isolate zero/negative prices, and flag non-product StockCodes.

## 4. Cleaning & Data Quality Decisions

Every removal is stored in a separate table with a documented reason. Nothing is silently dropped.

In [10]:
df_raw = df.copy()
print("Raw rows:", len(df_raw))

Raw rows: 541909


In [11]:
df["CustomerID"] = df["CustomerID"].astype("Int64").astype("string")

print(df["CustomerID"].dtype)
df["CustomerID"].head()

string


0    17850
1    17850
2    17850
3    17850
4    17850
Name: CustomerID, dtype: string

**Decision 4.1:** `CustomerID` was loaded as float64 because of missing values. It is an identifier, not a quantity, so it is converted to a string (via nullable `Int64` to avoid values like `17850.0`). Missing values stay as `<NA>`.

In [12]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour

df[["InvoiceDate", "Year", "Month", "DayOfWeek", "Hour"]].head()

,InvoiceDate,Year,Month,DayOfWeek,Hour
0,2010-12-01 08:26:00,2010,12,Wednesday,8
1,2010-12-01 08:26:00,2010,12,Wednesday,8
2,2010-12-01 08:26:00,2010,12,Wednesday,8
3,2010-12-01 08:26:00,2010,12,Wednesday,8
4,2010-12-01 08:26:00,2010,12,Wednesday,8


**Decision 4.2:** `InvoiceDate` was already parsed as datetime. Extracted `Year`, `Month`, `DayOfWeek`, and `Hour` to support seasonality and time-of-day analysis.

In [13]:
dup_mask = df.duplicated(keep="first")

df_duplicates = df[dup_mask].copy()
df = df[~dup_mask].copy()

print("Duplicate rows removed:", len(df_duplicates))
print("Rows remaining:", len(df))

Duplicate rows removed: 5268
Rows remaining: 536641


**Decision 4.3:** 5,268 fully duplicated rows (~1%) were removed, keeping the first occurrence. They are most likely double entries that would inflate revenue and quantity. Removed rows are stored in `df_duplicates`. Assumption: a small share could be genuine repeat purchases within the same minute.


In [14]:
guest_mask = df["CustomerID"].isna()

df_guest = df[guest_mask].copy()
df_customers = df[~guest_mask].copy()

print("Guest rows:", len(df_guest))
print("Guest invoices:", df_guest["InvoiceNo"].nunique())
print("Rows with CustomerID:", len(df_customers))
print("Customers:", df_customers["CustomerID"].nunique())

Guest rows: 135037
Guest invoices: 3710
Rows with CustomerID: 401604
Customers: 4372


**Decision 4.4:** Missing `CustomerID` rows form complete invoices (no invoice mixes known and missing IDs), so they are guest checkouts rather than random gaps. They are kept in `df` for sales, product, time, and country analysis, but split into `df_guest` and excluded from customer-level analysis (RFM and clustering) via `df_customers`, because they cannot be linked to a single customer.

In [15]:
df["IsCancellation"] = df["InvoiceNo"].astype(str).str.startswith("C")

df_cancel = df[df["IsCancellation"]].copy()

print("Cancellation lines:", len(df_cancel))
print("Cancellation invoices:", df_cancel["InvoiceNo"].nunique())
print("Cancellations with CustomerID:", df_cancel["CustomerID"].notna().sum())
print("Cancellations without CustomerID:", df_cancel["CustomerID"].isna().sum())
print("Cancelled value (GBP):", round((df_cancel["Quantity"] * df_cancel["UnitPrice"]).sum(), 2))

df_cancel.head()

Cancellation lines: 9251
Cancellation invoices: 3836
Cancellations with CustomerID: 8872
Cancellations without CustomerID: 379
Cancelled value (GBP): -893979.73


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,DayOfWeek,Hour,IsCancellation
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527,United Kingdom,2010,12,Wednesday,9,True
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311,United Kingdom,2010,12,Wednesday,9,True
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548,United Kingdom,2010,12,Wednesday,10,True
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom,2010,12,Wednesday,10,True
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom,2010,12,Wednesday,10,True


**Decision 4.5:** Cancellation lines (`InvoiceNo` starting with `C`) are flagged with `IsCancellation` and kept in `df`, since removing them would overstate revenue. They are copied to `df_cancel` for inspection and for matching against original invoices to compute net revenue.

In [16]:
cancel_c = df_cancel[df_cancel["CustomerID"].notna()].copy()
cancel_c["CustomerID"] = cancel_c["CustomerID"].astype(str)
cancel_c = cancel_c.reset_index(names="cancel_idx")

purchases = df.loc[
    (~df["IsCancellation"]) & (df["Quantity"] > 0) & df["CustomerID"].notna(),
    ["CustomerID", "StockCode", "InvoiceDate", "InvoiceNo", "Quantity", "UnitPrice"],
].copy()
purchases["CustomerID"] = purchases["CustomerID"].astype(str)
purchases["PurchaseDate"] = purchases["InvoiceDate"]
purchases = purchases.rename(columns={
    "InvoiceNo": "OrigInvoiceNo",
    "Quantity": "OrigQty",
    "UnitPrice": "OrigPrice",
})

df_matched = pd.merge_asof(
    cancel_c.sort_values("InvoiceDate"),
    purchases.sort_values("InvoiceDate"),
    on="InvoiceDate",
    by=["CustomerID", "StockCode"],
    direction="backward",
)

df_matched["Matched"] = df_matched["OrigInvoiceNo"].notna()
df_matched["DaysToCancel"] = (df_matched["InvoiceDate"] - df_matched["PurchaseDate"]).dt.days
df_matched["QtyFits"] = (-df_matched["Quantity"]) <= df_matched["OrigQty"]

n = len(df_matched)
print("Cancellation lines with CustomerID:", n)
print("Matched to an earlier purchase:", df_matched["Matched"].sum(), f"({df_matched['Matched'].mean():.1%})")
print("Matched and returned qty <= purchased qty:", (df_matched["Matched"] & df_matched["QtyFits"]).sum())
print("\nDays between purchase and cancellation (matched only):")
print(df_matched.loc[df_matched["Matched"], "DaysToCancel"].describe())

df_matched[~df_matched["Matched"]]["StockCode"].value_counts().head(10)

Cancellation lines with CustomerID: 8872
Matched to an earlier purchase: 7569 (85.3%)
Matched and returned qty <= purchased qty: 7411

Days between purchase and cancellation (matched only):
count    7569.000000
mean       25.458317
std        44.657324
min         0.000000
25%         4.000000
50%         9.000000
75%        24.000000
max       368.000000
Name: DaysToCancel, dtype: float64


StockCode
M        125
D         77
POST      57
22423     22
CRUK      16
21527     11
22941     10
71477      9
20914      9
35400      8
Name: count, dtype: int64

**Decision 4.5b:** 8,872 cancellation lines with a known `CustomerID` were matched to the most recent earlier purchase of the same product by the same customer. 7,569 (85.3%) were matched; 7,411 of them return a quantity not exceeding the purchased quantity. The median delay between purchase and cancellation is 9 days (max 368). Unmatched lines are mostly non-product codes (`M`, `D`, `POST`, `CRUK`) or likely purchases made before the dataset starts (Dec 2010); the latter is an assumption, not verified. Matching uses only the latest purchase, so returns spanning several purchases are partly unmatched. Cancellations stay in `df` and are netted against revenue.

In [17]:
neg_mask = (df["Quantity"] < 0) & (~df["IsCancellation"])
df_neg = df[neg_mask].copy()

print("Negative qty, no C prefix:", len(df_neg))
print("With CustomerID:", df_neg["CustomerID"].notna().sum())
print("Without CustomerID:", df_neg["CustomerID"].isna().sum())
print("UnitPrice == 0:", (df_neg["UnitPrice"] == 0).sum())
print("UnitPrice > 0:", (df_neg["UnitPrice"] > 0).sum())
print("Missing Description:", df_neg["Description"].isna().sum())
print("Value (GBP):", round((df_neg["Quantity"] * df_neg["UnitPrice"]).sum(), 2))

print("\nTop descriptions:")
print(df_neg["Description"].value_counts().head(10))

print("\nTop countries:")
print(df_neg["Country"].value_counts().head(5))

df_neg.head()

Negative qty, no C prefix: 1336
With CustomerID: 0
Without CustomerID: 1336
UnitPrice == 0: 1336
UnitPrice > 0: 0
Missing Description: 862
Value (GBP): 0.0

Top descriptions:
Description
check                     120
damages                    45
damaged                    42
?                          41
sold as set on dotcom      20
Damaged                    14
thrown away                 9
Unsaleable, destroyed.      9
??                          7
damages?                    5
Name: count, dtype: int64

Top countries:
Country
United Kingdom    1336
Name: count, dtype: int64


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,DayOfWeek,Hour,IsCancellation
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,<NA>,United Kingdom,2010,12,Wednesday,16,False
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,<NA>,United Kingdom,2010,12,Thursday,14,False
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,<NA>,United Kingdom,2010,12,Friday,15,False
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,<NA>,United Kingdom,2010,12,Friday,15,False
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,<NA>,United Kingdom,2010,12,Friday,15,False


In [18]:
df_inventory_adj = df[neg_mask].copy()
df = df[~neg_mask].copy()

print("Inventory adjustment rows isolated:", len(df_inventory_adj))
print("Rows remaining:", len(df))

Inventory adjustment rows isolated: 1336
Rows remaining: 535305


**Decision 4.6:** 1,336 lines with negative `Quantity` and no `C` prefix were isolated into `df_inventory_adj` and removed from `df`. All have `UnitPrice` = 0, no `CustomerID`, and are UK-only; 862 have no `Description` and the rest carry notes such as "damaged", "check", or "thrown away", which indicates manual stock adjustments rather than sales. They have zero revenue impact, but would distort unit counts and invoice counts.

In [19]:
price_mask = df["UnitPrice"] <= 0
df_price = df[price_mask].copy()

print("Rows with UnitPrice <= 0:", len(df_price))
print("UnitPrice == 0:", (df_price["UnitPrice"] == 0).sum())
print("UnitPrice < 0:", (df_price["UnitPrice"] < 0).sum())
print("With CustomerID:", df_price["CustomerID"].notna().sum())
print("Without CustomerID:", df_price["CustomerID"].isna().sum())
print("Missing Description:", df_price["Description"].isna().sum())
print("Quantity > 0:", (df_price["Quantity"] > 0).sum())
print("Quantity < 0:", (df_price["Quantity"] < 0).sum())
print("Cancellation lines:", df_price["IsCancellation"].sum())

print("\nQuantity summary:")
print(df_price["Quantity"].describe())

print("\nTop descriptions:")
print(df_price["Description"].value_counts().head(10))

print("\nNegative price rows:")
print(df_price[df_price["UnitPrice"] < 0])

df_price.head()

Rows with UnitPrice <= 0: 1176
UnitPrice == 0: 1174
UnitPrice < 0: 2
With CustomerID: 40
Without CustomerID: 1136
Missing Description: 592
Quantity > 0: 1176
Quantity < 0: 0
Cancellation lines: 0

Quantity summary:
count     1176.000000
mean        61.733844
std        454.062217
min          1.000000
25%          1.000000
50%          3.000000
75%         20.000000
max      12540.000000
Name: Quantity, dtype: float64

Top descriptions:
Description
check                              39
found                              25
adjustment                         14
FRENCH BLUE METAL DOOR SIGN 1       9
amazon                              8
Found                               8
FRENCH BLUE METAL DOOR SIGN 8       8
RECIPE BOX PANTRY YELLOW DESIGN     7
FRENCH BLUE METAL DOOR SIGN 3       7
OWL DOORSTOP                        7
Name: count, dtype: int64

Negative price rows:
       InvoiceNo StockCode      Description  Quantity         InvoiceDate  \
299983   A563186         B  Adjust bad deb

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,DayOfWeek,Hour,IsCancellation
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,<NA>,United Kingdom,2010,12,Wednesday,11,False
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,<NA>,United Kingdom,2010,12,Wednesday,14,False
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,<NA>,United Kingdom,2010,12,Wednesday,14,False
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,<NA>,United Kingdom,2010,12,Wednesday,14,False
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,<NA>,United Kingdom,2010,12,Wednesday,14,False


In [20]:
price_mask = df["UnitPrice"] <= 0
df_price_zero_neg = df[price_mask].copy()
df = df[~price_mask].copy()

print("Zero/negative price rows isolated:", len(df_price_zero_neg))
print("Rows remaining:", len(df))

Zero/negative price rows isolated: 1176
Rows remaining: 534129


**Decision 4.7:** 1,176 lines with `UnitPrice` <= 0 were isolated into `df_price_zero_neg` and removed from `df`. 1,174 have `UnitPrice` = 0 (all with positive `Quantity`; 1,136 without `CustomerID`; 592 without `Description`), with notes such as "check", "found", "adjustment" suggesting manual stock corrections, plus some real products that left stock at zero price (gifts, samples, or pricing omissions; cannot be determined from the data). 2 lines (`A563186`, `A563187`) are "Adjust bad debt" entries at -11,062.06, accounting adjustments rather than sales. None of these represent real revenue, and keeping them would inflate unit counts or distort revenue.

In [21]:
pattern_ok = df["StockCode"].astype(str).str.match(r"^\d{5}[A-Za-z]{0,2}$")
odd = df[~pattern_ok].copy()
odd["Revenue"] = odd["Quantity"] * odd["UnitPrice"]

summary = (
    odd.groupby("StockCode")
    .agg(
        lines=("InvoiceNo", "size"),
        revenue=("Revenue", "sum"),
        top_description=("Description", lambda s: s.mode().iloc[0] if s.notna().any() else None),
    )
    .sort_values("lines", ascending=False)
)

print("Lines with non-standard StockCode:", len(odd))
print("Distinct codes:", summary.shape[0])
summary.head(25)

Lines with non-standard StockCode: 2957
Distinct codes: 24


,lines,revenue,top_description
StockCode,,,
POST,1252,66230.640,POSTAGE
DOT,707,206245.480,DOTCOM POSTAGE
M,560,-69034.190,Manual
C2,143,6986.000,CARRIAGE
D,77,-5696.220,Discount
S,62,-3039.650,SAMPLES
BANK CHARGES,37,-7175.639,Bank Charges
AMAZONFEE,34,-221520.500,AMAZON FEE
CRUK,16,-7933.430,CRUK Commission


In [22]:
non_product_codes = [
    "POST", "DOT", "C2", "M", "m", "D", "S",
    "BANK CHARGES", "AMAZONFEE", "CRUK", "B",
]
np_mask = df["StockCode"].isin(non_product_codes)

df_non_product = df[np_mask].copy()
df = df[~np_mask].copy()

print("Non-product lines isolated:", len(df_non_product))
print("Rows remaining:", len(df))
print("\nIsolated revenue by code:")
print(
    (df_non_product["Quantity"] * df_non_product["UnitPrice"])
    .groupby(df_non_product["StockCode"]).sum().round(2)
)

Non-product lines isolated: 2890
Rows remaining: 531239

Isolated revenue by code:
StockCode
AMAZONFEE      -221520.50
B                11062.06
BANK CHARGES     -7175.64
C2                6986.00
CRUK             -7933.43
D                -5696.22
DOT             206245.48
M               -69034.19
POST             66230.64
S                -3039.65
m                    2.55
dtype: float64


**Decision 4.8:** 2,890 lines with non-product `StockCode` values were isolated into `df_non_product` and removed from `df`: shipping (`POST`, `DOT`, `C2`), manual entries (`M`, `m`), discounts (`D`), samples (`S`), bank and marketplace fees (`BANK CHARGES`, `AMAZONFEE`, `CRUK`), and the bad-debt adjustment counterpart (`B`). `AMAZONFEE` alone is about -221K GBP, so these lines are kept separately in case total financial revenue is needed. Gift vouchers (`gift_*`, 31 lines, about 686 GBP) were kept as regular sales because customers actually paid for them. Real products with unusual codes (`DCGS*`, `PADS`) were also kept.

In [23]:
sales_ = df[(~df["IsCancellation"]) & df["CustomerID"].notna()].reset_index(names="sale_idx")
canc_ = df[df["IsCancellation"] & df["CustomerID"].notna()].reset_index(names="canc_idx")
canc_["Quantity"] = -canc_["Quantity"]

pairs = canc_.merge(
    sales_,
    on=["CustomerID", "StockCode", "Quantity", "UnitPrice"],
    suffixes=("_c", "_s"),
)
pairs["minutes"] = (pairs["InvoiceDate_c"] - pairs["InvoiceDate_s"]).dt.total_seconds() / 60
pairs = pairs[(pairs["minutes"] >= 0) & (pairs["minutes"] <= 60)]

# Look at large ones first, to decide the threshold
print("Same-hour exact reversals:", len(pairs))
print(pairs["Quantity"].describe())
pairs.nlargest(10, "Quantity")[
    ["CustomerID", "StockCode", "Quantity", "UnitPrice", "InvoiceNo_s", "InvoiceNo_c", "minutes"]
]

Same-hour exact reversals: 468
count      468.000000
mean       377.933761
std       5070.206143
min          1.000000
25%          5.000000
50%         15.000000
75%         48.000000
max      80995.000000
Name: Quantity, dtype: float64


,CustomerID,StockCode,Quantity,UnitPrice,InvoiceNo_s,InvoiceNo_c,minutes
5225,16446,23843,80995,2.08,581483,C581484,12.0
405,12346,23166,74215,1.04,541431,C541433,16.0
4294,14607,22102,720,0.39,572324,C572343,48.0
4295,14607,22102,720,0.39,572340,C572343,4.0
4292,14607,22103,468,0.39,572324,C572343,48.0
4293,14607,22103,468,0.39,572340,C572343,4.0
4118,12901,21787,360,0.85,571259,C571262,33.0
4119,12901,21787,360,0.85,571261,C571262,2.0
4296,14607,22101,324,0.39,572324,C572343,48.0
4297,14607,22101,324,0.39,572340,C572343,4.0


In [24]:
big_pairs = pairs[pairs["Quantity"] > 10000]
rev_invoices = set(big_pairs["InvoiceNo_s"].astype(str)) | set(big_pairs["InvoiceNo_c"].astype(str))
rev_mask = df["InvoiceNo"].astype(str).isin(rev_invoices)

df_reversed = df[rev_mask].copy()
df = df[~rev_mask].copy()

print("Invoices isolated:", sorted(rev_invoices))
print("Rows isolated:", len(df_reversed))
print("Net of isolated rows (GBP):",
      round((df_reversed["Quantity"] * df_reversed["UnitPrice"]).sum(), 2))
print("Rows remaining:", len(df))

Invoices isolated: ['541431', '581483', 'C541433', 'C581484']
Rows isolated: 4
Net of isolated rows (GBP): 0.0
Rows remaining: 531235


**Decision 4.9:** Sales fully cancelled within an hour by the same customer (same product, quantity, and price) were detected programmatically (468 pairs). Most are small genuine returns (median 15 units) and are kept. Only two orders far above the rest (74,215 and 80,995 units; the next largest is 720) were isolated with their cancellations into `df_reversed`. Net effect is zero, but they distorted unit rankings, gross sales (by ~246K GBP), and the cancellation rate.

In [25]:
df_guest = df[df["CustomerID"].isna()].copy()
df_customers = df[df["CustomerID"].notna()].copy()
df_cancel = df[df["IsCancellation"]].copy()

# Consistency checks
assert len(df_guest) + len(df_customers) == len(df)
assert df["Quantity"].ne(0).all()
assert (df["UnitPrice"] > 0).all()
assert ((df["Quantity"] < 0) == df["IsCancellation"]).all()

steps = pd.DataFrame(
    [
        ("Raw data", len(df_raw)),
        ("Duplicates removed (4.3)", -len(df_duplicates)),
        ("Inventory adjustments isolated (4.6)", -len(df_inventory_adj)),
        ("Zero/negative price isolated (4.7)", -len(df_price_zero_neg)),
        ("Non-product codes isolated (4.8)", -len(df_non_product)),
        ("Fully reversed orders isolated (4.9)", -len(df_reversed)),
        ("Final df", len(df)),
    ],
    columns=["step", "rows"],
)
steps["check"] = ""
steps.loc[steps.index[-1], "check"] = (
    "OK" if steps["rows"].iloc[:-1].sum() == len(df) else "MISMATCH"
)
display(steps)

print("Final rows:", len(df), f"({len(df) / len(df_raw):.1%} of raw)")
print("Invoices:", df["InvoiceNo"].nunique())
print("Customers:", df_customers["CustomerID"].nunique())
print("Guest rows:", len(df_guest), "| Customer rows:", len(df_customers))
print("Cancellation lines:", len(df_cancel), "| value (GBP):",
      round((df_cancel["Quantity"] * df_cancel["UnitPrice"]).sum(), 2))

,step,rows,check
0,Raw data,541909,
1,Duplicates removed (4.3),-5268,
2,Inventory adjustments isolated (4.6),-1336,
3,Zero/negative price isolated (4.7),-1176,
4,Non-product codes isolated (4.8),-2890,
5,Fully reversed orders isolated (4.9),-4,
6,Final df,531235,OK


Final rows: 531235 (98.0% of raw)
Invoices: 23191
Customers: 4361
Guest rows: 131580 | Customer rows: 399655
Cancellation lines: 8666 | value (GBP): -230247.96


### Cleaning Summary

| Step | Rows removed |
|---|---|
| Duplicates (4.3) | 5,268 |
| Inventory adjustments (4.6) | 1,336 |
| Zero/negative prices (4.7) | 1,176 |
| Non-product codes (4.8) | 2,890 |
| Fully reversed orders (4.9) | 4 |
| **Total** | **10,674 (2.0%)** |

**Final dataset:** 531,235 rows, 23,191 invoices, 4,361 customers. Guest checkouts (131,580 rows) stay in `df` for sales analysis but are excluded from customer-level analysis via `df_customers` (399,655 rows).

**Cancellations:** 8,666 lines worth about -230K GBP remain in `df` (flagged by `IsCancellation`) and will be netted against revenue. About 85% of cancellations with a known customer were matched to an earlier purchase (4.5b, computed before the reversed orders were isolated).

**Removed rows are stored in:** `df_duplicates`, `df_inventory_adj`, `df_price_zero_neg`, `df_non_product`, `df_reversed`.

## 5. Feature Engineering

Each row is an invoice line, not a full order. Revenue is computed per line; cancellations carry negative revenue, so summing it gives net revenue.

In [26]:
df["TotalLineRevenue"] = df["Quantity"] * df["UnitPrice"]
df["Is_UK"] = (df["Country"] == "United Kingdom").astype(int)
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)

# Rebuild subsets so they include the new columns
df_guest = df[df["CustomerID"].isna()].copy()
df_customers = df[df["CustomerID"].notna()].copy()
df_cancel = df[df["IsCancellation"]].copy()

gross = df.loc[~df["IsCancellation"], "TotalLineRevenue"].sum()
cancelled = df.loc[df["IsCancellation"], "TotalLineRevenue"].sum()

print(f"Gross sales (GBP):   {gross:,.2f}")
print(f"Cancellations (GBP): {cancelled:,.2f}")
print(f"Net revenue (GBP):   {df['TotalLineRevenue'].sum():,.2f}")
print(f"Cancelled share of gross: {-cancelled / gross:.1%}")
print(f"UK share of rows: {df['Is_UK'].mean():.1%}")

df[["InvoiceNo", "Quantity", "UnitPrice", "TotalLineRevenue", "Country", "Is_UK", "YearMonth"]].head()

Gross sales (GBP):   10,002,251.93
Cancellations (GBP): -230,247.96
Net revenue (GBP):   9,772,003.97
Cancelled share of gross: 2.3%
UK share of rows: 91.5%


,InvoiceNo,Quantity,UnitPrice,TotalLineRevenue,Country,Is_UK,YearMonth
0,536365,6,2.55,15.30,United Kingdom,1,2010-12
1,536365,6,3.39,20.34,United Kingdom,1,2010-12
2,536365,8,2.75,22.00,United Kingdom,1,2010-12
3,536365,6,3.39,20.34,United Kingdom,1,2010-12
4,536365,6,3.39,20.34,United Kingdom,1,2010-12


**Decision 5:** Added three features: `TotalLineRevenue` (`Quantity * UnitPrice`; negative for cancellations, so its sum is net revenue), `Is_UK` (1 for United Kingdom, 0 otherwise), and `YearMonth` (for monthly analysis). Subsets `df_guest`, `df_customers`, and `df_cancel` were rebuilt to include them.

**Key figures (product revenue, excluding shipping and fees):**
- Gross sales: 10,002,252 GBP
- Cancellations: -230,248 GBP (2.3% of gross)
- Net revenue: 9,772,004 GBP
- UK accounts for 91.5% of rows (share of revenue to be computed in EDA)

Note: some cancellations may refer to purchases made before the dataset starts, so the cancellation rate is approximate.

## 6. Exploratory Data Analysis


In [27]:
import plotly.express as px

sales = df[~df["IsCancellation"]].copy()

# One StockCode can have several descriptions: use the most frequent one as its name
names = sales.groupby("StockCode")["Description"].agg(
    lambda s: s.mode().iloc[0] if s.notna().any() else None
)

products = (
    sales.groupby("StockCode")
    .agg(units=("Quantity", "sum"), revenue=("TotalLineRevenue", "sum"))
    .join(names.rename("name"))
    .reset_index()
)

print("Distinct products sold:", len(products))

top_units = products.nlargest(10, "units").sort_values("units")
top_rev = products.nlargest(10, "revenue").sort_values("revenue")

fig1 = px.bar(top_units, x="units", y="name", orientation="h",
              title="Top 10 products by units sold",
              labels={"units": "Units sold", "name": ""})
fig1.show()

fig2 = px.bar(top_rev, x="revenue", y="name", orientation="h",
              title="Top 10 products by revenue (GBP)",
              labels={"revenue": "Revenue (GBP)", "name": ""})
fig2.show()

print("Top 10 by units that are also in top 10 by revenue:",
      len(set(top_units["StockCode"]) & set(top_rev["StockCode"])))

Distinct products sold: 3912


Top 10 by units that are also in top 10 by revenue: 5


In [28]:
big = df[df["StockCode"].isin(
    products.nlargest(2, "units")["StockCode"]
)].copy()

big = big[big["Quantity"].abs() > 10000]
big[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "InvoiceDate", "CustomerID"]]

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,InvoiceDate,CustomerID


In [29]:
cust = (
    df_customers.groupby("CustomerID")
    .agg(
        net_revenue=("TotalLineRevenue", "sum"),
        country=("Country", lambda s: s.mode().iloc[0]),
    )
)
cust["orders"] = (
    df_customers[~df_customers["IsCancellation"]]
    .groupby("CustomerID")["InvoiceNo"].nunique()
)
cust["orders"] = cust["orders"].fillna(0).astype(int)
cust = cust.reset_index()

print("Customers:", len(cust))
print("Customers with zero sales invoices:", (cust["orders"] == 0).sum())

top_rev = cust.nlargest(10, "net_revenue").sort_values("net_revenue")
top_ord = cust.nlargest(10, "orders").sort_values("orders")

fig1 = px.bar(top_rev, x="net_revenue", y="CustomerID", orientation="h",
              title="Top 10 customers by net revenue (GBP)",
              labels={"net_revenue": "Net revenue (GBP)", "CustomerID": "CustomerID"})
fig1.update_yaxes(type="category")
fig1.show()

fig2 = px.bar(top_ord, x="orders", y="CustomerID", orientation="h",
              title="Top 10 customers by number of orders",
              labels={"orders": "Orders (unique invoices)", "CustomerID": "CustomerID"})
fig2.update_yaxes(type="category")
fig2.show()

top10_share = cust.nlargest(10, "net_revenue")["net_revenue"].sum() / cust["net_revenue"].sum()
print(f"Top 10 customers share of customer net revenue: {top10_share:.1%}")
print("In both top 10 lists:", len(set(top_rev["CustomerID"]) & set(top_ord["CustomerID"])))

cust.nlargest(10, "net_revenue")

Customers: 4361
Customers with zero sales invoices: 28


Top 10 customers share of customer net revenue: 16.6%
In both top 10 lists: 3


,CustomerID,net_revenue,country,orders
1698,14646,278778.02,Netherlands,72
4222,18102,259657.30,United Kingdom,60
3748,17450,189575.53,United Kingdom,46
1890,14911,128768.24,EIRE,198
54,12415,123638.18,Australia,20
1340,14156,113685.77,EIRE,54
3791,17511,88138.20,United Kingdom,31
3194,16684,65920.12,United Kingdom,28
1000,13694,62961.54,United Kingdom,50
2714,16029,60369.93,United Kingdom,62


### Insights 6.1 (products and customers)

1. **Top products by units and revenue differ.** Only 6 of the top 10 products by units are also in the top 10 by revenue. `REGENCY CAKESTAND 3 TIER` leads revenue (~174K GBP) while `POPCORN HOLDER` leads units (~57K), reflecting price differences.
2. **Two fully reversed orders (74,215 and 80,995 units) distorted the first rankings.** After isolating them (4.9), gross sales fell by ~246K GBP and the cancellation rate dropped from 4.6% to 2.3%; net revenue was unchanged.
3. **Customer revenue is concentrated at the top.** The top 10 customers account for 16.6% of registered-customer net revenue (4,361 customers). The top two (14646, 18102) exceed 255K GBP each; the third is ~190K.
4. **Order count and revenue identify different customers.** Only 3 customers appear in both top 10 lists. Customer 14911 places 198 orders for ~129K GBP, while 12415 generates ~124K GBP from only 20 orders (~6.2K GBP per order), a pattern consistent with bulk buying (not verifiable without customer type data).
5. **Several top customers are outside the UK** (Netherlands, EIRE, Australia), although the UK is 91.5% of rows.
6. **28 customers have no sales invoice in the period**, only cancellations, which may relate to purchases made before December 2010 (assumption).

All findings are descriptive; no causal claims are made.

In [30]:
sales_only = df[~df["IsCancellation"]]

# Monthly
monthly = df.groupby("YearMonth").agg(net_revenue=("TotalLineRevenue", "sum")).reset_index()
monthly["invoices"] = monthly["YearMonth"].map(
    sales_only.groupby("YearMonth")["InvoiceNo"].nunique()
)
monthly["note"] = np.where(monthly["YearMonth"] == "2011-12", "incomplete (to Dec 9)", "")

fig1 = px.bar(monthly, x="YearMonth", y="net_revenue", color="note",
              title="Monthly net revenue (GBP)",
              labels={"net_revenue": "Net revenue (GBP)", "YearMonth": "Month", "note": ""})
fig1.show()

# Weekday
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = df.groupby("DayOfWeek").agg(net_revenue=("TotalLineRevenue", "sum")).reindex(order).reset_index()
weekday["invoices"] = weekday["DayOfWeek"].map(sales_only.groupby("DayOfWeek")["InvoiceNo"].nunique())

fig2 = px.bar(weekday, x="DayOfWeek", y="net_revenue",
              title="Net revenue by weekday (GBP)",
              labels={"net_revenue": "Net revenue (GBP)", "DayOfWeek": ""})
fig2.show()

# Hour
hourly = df.groupby("Hour").agg(net_revenue=("TotalLineRevenue", "sum")).reset_index()

fig3 = px.bar(hourly, x="Hour", y="net_revenue",
              title="Net revenue by hour of day (GBP)",
              labels={"net_revenue": "Net revenue (GBP)", "Hour": "Hour"})
fig3.show()

print("Best month:", monthly.loc[monthly["net_revenue"].idxmax(), ["YearMonth", "net_revenue"]].tolist())
print("Weekday with no sales:", weekday.loc[weekday["net_revenue"].isna(), "DayOfWeek"].tolist())
print("\nMonthly table:")
print(monthly.round(0).to_string(index=False))
print("\nWeekday table:")
print(weekday.round(0).to_string(index=False))

Best month: ['2011-11', np.float64(1427124.54)]
Weekday with no sales: ['Saturday']

Monthly table:
YearMonth  net_revenue  invoices                  note
  2010-12     758167.0      1550                      
  2011-01     578914.0      1080                      
  2011-02     499531.0      1093                      
  2011-03     679338.0      1440                      
  2011-04     482154.0      1235                      
  2011-05     731088.0      1668                      
  2011-06     723970.0      1525                      
  2011-07     676845.0      1452                      
  2011-08     701382.0      1339                      
  2011-09    1011365.0      1818                      
  2011-10    1061718.0      2005                      
  2011-11    1427125.0      2751                      
  2011-12     440408.0       815 incomplete (to Dec 9)

Weekday table:
DayOfWeek  net_revenue  invoices
   Monday    1622366.0    3076.0
  Tuesday    1968174.0    3508.0
Wednesday    17

In [31]:
raw_days = df_raw["InvoiceDate"].dt.day_name().value_counts()
print(raw_days)

InvoiceDate
Thursday     103857
Tuesday      101808
Monday        95111
Wednesday     94565
Friday        82193
Sunday        64375
Name: count, dtype: int64


### Insights 6.2 (time)

7. **Revenue rises sharply toward year-end.** Monthly net revenue ranges from ~482K to ~731K GBP between Jan and Aug 2011, then reaches ~1.01M (Sep), ~1.06M (Oct), and a peak of ~1.43M in Nov 2011.
8. **December 2011 is incomplete** (data ends Dec 9), so its ~440K GBP is not comparable to a full month. It already holds 815 invoices in 9 days vs 1,550 for all of Dec 2010.
9. **No transactions are recorded on Saturdays**, neither in the raw data nor after cleaning (verified on `df_raw`). The reason is not available in the data, so Saturday is treated as missing, not as a weak day.
10. **Thursday (~2.08M GBP) and Tuesday (~1.97M GBP) are the strongest weekdays**; Sunday is the weakest active day (~0.79M GBP).
11. **Sales concentrate between 09:00 and 16:00**, with peaks around 10:00, 12:00, and 15:00, and very little activity before 08:00 or after 18:00 (read from the chart).

*Findings are descriptive. Seasonality is based on one year of data, so it cannot be separated from other effects.*

In [32]:
sales_only = df[~df["IsCancellation"]]

countries = df.groupby("Country").agg(net_revenue=("TotalLineRevenue", "sum"))
countries["orders"] = sales_only.groupby("Country")["InvoiceNo"].nunique()
countries["customers"] = df_customers.groupby("Country")["CustomerID"].nunique()
countries = countries.fillna(0).astype({"orders": int, "customers": int}).reset_index()

countries["revenue_share"] = countries["net_revenue"] / countries["net_revenue"].sum()
countries["aov"] = countries["net_revenue"] / countries["orders"].replace(0, np.nan)

uk = countries[countries["Country"] == "United Kingdom"].iloc[0]
print("Countries:", len(countries))
print(f"UK share of net revenue: {uk['revenue_share']:.1%}")
print(f"UK share of orders: {uk['orders'] / countries['orders'].sum():.1%}")
print(f"UK share of customers: {uk['customers'] / countries['customers'].sum():.1%}")

top_c = countries[countries["Country"] != "United Kingdom"].nlargest(10, "net_revenue").sort_values("net_revenue")
fig1 = px.bar(top_c, x="net_revenue", y="Country", orientation="h",
              title="Top 10 non-UK countries by net revenue (GBP)",
              labels={"net_revenue": "Net revenue (GBP)", "Country": ""})
fig1.show()

fig2 = px.bar(countries.nlargest(10, "customers").sort_values("customers"),
              x="customers", y="Country", orientation="h",
              title="Top 10 countries by number of customers",
              labels={"customers": "Customers", "Country": ""})
fig2.show()

countries.sort_values("net_revenue", ascending=False).head(10).round(3)

Countries: 38
UK share of net revenue: 84.8%
UK share of orders: 90.5%
UK share of customers: 90.2%


,Country,net_revenue,orders,customers,revenue_share,aov
36,United Kingdom,8282076.173,17899,3941,0.848,462.712
24,Netherlands,283479.540,93,9,0.029,3048.167
10,EIRE,259380.020,282,3,0.027,919.787
14,Germany,200619.660,443,95,0.021,452.866
13,France,182076.600,382,87,0.019,476.640
0,Australia,136922.500,56,9,0.014,2445.045
33,Switzerland,52483.050,50,21,0.005,1049.661
31,Spain,51746.650,88,30,0.005,588.030
3,Belgium,36662.960,98,25,0.004,374.112
20,Japan,35419.790,19,8,0.004,1864.199


In [33]:
renamed = {"EIRE": "Ireland", "RSA": "South Africa"}
excluded = ["Unspecified", "European Community", "Channel Islands"]

map_df = countries.copy()
map_df["Country"] = map_df["Country"].replace(renamed)
map_df = map_df[~map_df["Country"].isin(excluded)]

# UK excluded so other countries are visible; log scale handles the large gaps
map_nonuk = map_df[map_df["Country"] != "United Kingdom"].copy()
map_nonuk["log_revenue"] = np.log10(map_nonuk["net_revenue"].clip(lower=1))

fig_map = px.choropleth(
    map_nonuk, locations="Country", locationmode="country names",
    color="log_revenue", hover_name="Country",
    hover_data={"net_revenue": ":,.0f", "customers": True, "orders": True, "log_revenue": False},
    title="Net revenue by country, excluding UK (log10 scale, GBP)",
    labels={"log_revenue": "log10(revenue)"},
)
fig_map.show()

placed = set(map_nonuk["Country"]) | {"United Kingdom"}
not_placed = sorted(c for c in countries["Country"] if renamed.get(c, c) not in placed)
print("Not placed on map:", not_placed)

C:\Users\elmoulll\AppData\Local\Temp\ipykernel_36524\2020028018.py:12: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig_map = px.choropleth(


Not placed on map: ['Channel Islands', 'European Community', 'Unspecified']


**Map note:** `Channel Islands`, `European Community`, and `Unspecified` are not real country polygons and are not shown on the map (they remain in the tables). `EIRE` and `RSA` were renamed to Ireland and South Africa for matching. The UK is excluded from the map so other countries are visible, and the color scale is log10, so color differences reflect orders of magnitude, not linear gaps.

### Insights 6.3 (countries)

12. **The UK dominates but less in revenue than in volume.** It accounts for 90.5% of orders and 90.2% of registered customers, but 84.8% of net revenue (~8.28M GBP), because several non-UK countries have much higher average order values.
13. **Several non-UK countries rank high on revenue with very few customers.** Netherlands (~283K GBP, 9 customers, AOV ~3,048 GBP), EIRE (~259K GBP, 3 customers) and Australia (~137K GBP, 9 customers, AOV ~2,445 GBP) depend on a handful of accounts, so their rankings are fragile.
14. **Germany and France show a broader customer base.** They generate ~201K and ~182K GBP from 95 and 87 customers, with AOV (~453 and ~477 GBP) close to the UK's (~463 GBP).

All figures are product revenue (shipping and fees excluded) and descriptive only.

In [34]:
import plotly.graph_objects as go

# Daily net revenue
daily = df.groupby(df["InvoiceDate"].dt.date).agg(net_revenue=("TotalLineRevenue", "sum")).reset_index()
daily["InvoiceDate"] = pd.to_datetime(daily["InvoiceDate"])
daily["YearMonth"] = daily["InvoiceDate"].dt.to_period("M").astype(str)

# Average daily revenue per month (over days with sales)
m = daily.groupby("YearMonth").agg(
    total=("net_revenue", "sum"),
    active_days=("net_revenue", "size"),
).reset_index()
m["avg_daily"] = m["total"] / m["active_days"]
m["note"] = np.where(m["YearMonth"] == "2011-12", "incomplete (to Dec 9)", "")

fig1 = px.bar(m, x="YearMonth", y="avg_daily", color="note",
              title="Average daily net revenue by month (GBP, active days only)",
              labels={"avg_daily": "Avg daily net revenue (GBP)", "YearMonth": "Month", "note": ""})
fig1.show()

# Cumulative revenue
daily = daily.sort_values("InvoiceDate")
daily["cumulative"] = daily["net_revenue"].cumsum()
fig2 = px.line(daily, x="InvoiceDate", y="cumulative",
               title="Cumulative net revenue (GBP)",
               labels={"cumulative": "Cumulative net revenue (GBP)", "InvoiceDate": "Date"})
fig2.show()

# Heatmap: month x weekday
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat = (df.pivot_table(index="DayOfWeek", columns="YearMonth",
                       values="TotalLineRevenue", aggfunc="sum")
          .reindex(order))
fig3 = px.imshow(heat, aspect="auto", color_continuous_scale="Blues",
                 title="Net revenue heatmap: weekday x month (GBP)",
                 labels={"color": "Net revenue (GBP)", "x": "Month", "y": ""})
fig3.show()

# Fair comparison: Oct/Nov 2011 vs Jan-Aug 2011 baseline
base = m[m["YearMonth"].between("2011-01", "2011-08")]
q4 = m[m["YearMonth"].isin(["2011-10", "2011-11"])]
print(f"Jan-Aug 2011 avg monthly revenue: {base['total'].mean():,.0f}")
print(f"Oct-Nov 2011 avg monthly revenue: {q4['total'].mean():,.0f}  "
      f"({q4['total'].mean() / base['total'].mean():.2f}x baseline)")
print(f"Jan-Aug 2011 avg daily revenue:   {base['avg_daily'].mean():,.0f}")
print(f"Oct-Nov 2011 avg daily revenue:   {q4['avg_daily'].mean():,.0f}  "
      f"({q4['avg_daily'].mean() / base['avg_daily'].mean():.2f}x baseline)")

m.round(0)

Jan-Aug 2011 avg monthly revenue: 634,153
Oct-Nov 2011 avg monthly revenue: 1,244,421  (1.96x baseline)
Jan-Aug 2011 avg daily revenue:   25,394
Oct-Nov 2011 avg daily revenue:   47,862  (1.88x baseline)


,YearMonth,total,active_days,avg_daily,note
0,2010-12,758167.0,20,37908.0,
1,2011-01,578914.0,24,24121.0,
2,2011-02,499531.0,24,20814.0,
3,2011-03,679338.0,27,25161.0,
4,2011-04,482154.0,21,22960.0,
5,2011-05,731088.0,25,29244.0,
6,2011-06,723970.0,26,27845.0,
7,2011-07,676845.0,26,26033.0,
8,2011-08,701382.0,26,26976.0,
9,2011-09,1011365.0,26,38899.0,


### Insights 6.4 (seasonality)

15. **Q4 revenue is roughly double the baseline.** Oct-Nov 2011 averages ~1.24M GBP per month vs ~634K for Jan-Aug 2011 (1.96x); on an average-daily basis the ratio is 1.88x (~47.9K vs ~25.4K GBP per active day).
16. **The rise starts in September, before Q4.** Average daily revenue goes from ~26-27K (Jul-Aug) to ~38.9K in Sep, ~40.8K in Oct and ~54.9K in Nov 2011.
17. **Early December looks as strong as November, but on very little data.** Dec 2011 shows ~55.1K GBP per active day over only 8 days, so it is not strong evidence; Dec 2010 (20 active days) averaged ~37.9K.

Findings are descriptive. With a single year of data, seasonality cannot be separated from growth or promotions, and Q4 2010 is not available for comparison (data starts Dec 1, 2010).

In [35]:
canc = df[df["IsCancellation"]]
sales_only = df[~df["IsCancellation"]]

gross = sales_only["TotalLineRevenue"].sum()
canc_val = -canc["TotalLineRevenue"].sum()
n_sales_inv = sales_only["InvoiceNo"].nunique()
n_canc_inv = canc["InvoiceNo"].nunique()

print(f"Cancellation lines: {len(canc):,} ({len(canc) / len(df):.1%} of lines)")
print(f"Cancellation invoices: {n_canc_inv:,} vs sales invoices: {n_sales_inv:,} "
      f"({n_canc_inv / n_sales_inv:.1%})")
print(f"Cancelled value: {canc_val:,.0f} GBP ({canc_val / gross:.1%} of gross)")
print(f"Customers with at least one cancellation: "
      f"{canc['CustomerID'].nunique():,} of {df_customers['CustomerID'].nunique():,}")

# Monthly cancellation rate (by value)
mm = pd.DataFrame({
    "gross": sales_only.groupby("YearMonth")["TotalLineRevenue"].sum(),
    "cancelled": -canc.groupby("YearMonth")["TotalLineRevenue"].sum(),
}).fillna(0).reset_index()
mm["rate"] = mm["cancelled"] / mm["gross"]

fig1 = px.line(mm, x="YearMonth", y="rate", markers=True,
               title="Monthly cancellation rate (cancelled value / gross sales)",
               labels={"rate": "Cancellation rate", "YearMonth": "Month"})
fig1.update_yaxes(tickformat=".1%")
fig1.show()

# Products: cancelled units vs sold units
p_sold = sales_only.groupby("StockCode")["Quantity"].sum().rename("sold_units")
p_canc = (-canc.groupby("StockCode")["Quantity"].sum()).rename("cancelled_units")
p_val = (-canc.groupby("StockCode")["TotalLineRevenue"].sum()).rename("cancelled_value")
p_name = sales_only.groupby("StockCode")["Description"].agg(
    lambda s: s.mode().iloc[0] if s.notna().any() else None).rename("name")
prod = pd.concat([p_sold, p_canc, p_val, p_name], axis=1).fillna({"cancelled_units": 0, "cancelled_value": 0})
prod = prod.dropna(subset=["sold_units"]).reset_index()
prod["return_rate"] = prod["cancelled_units"] / prod["sold_units"]

top_cp = prod.nlargest(10, "cancelled_value").sort_values("cancelled_value")
fig2 = px.bar(top_cp, x="cancelled_value", y="name", orientation="h",
              title="Top 10 products by cancelled value (GBP)",
              labels={"cancelled_value": "Cancelled value (GBP)", "name": ""})
fig2.show()

# Scatter: volume vs return rate (products with at least 500 units sold)
big = prod[prod["sold_units"] >= 500]
fig3 = px.scatter(big, x="sold_units", y="return_rate", hover_name="name",
                  log_x=True, title="Units sold vs return rate (products with 500+ units sold)",
                  labels={"sold_units": "Units sold (log)", "return_rate": "Return rate (units)"})
fig3.update_yaxes(tickformat=".0%")
fig3.show()

# Customers
cc = (-canc.groupby("CustomerID")["TotalLineRevenue"].sum()).rename("cancelled_value")
cs = df_customers[~df_customers["IsCancellation"]].groupby("CustomerID")["TotalLineRevenue"].sum().rename("gross")
cust_c = pd.concat([cc, cs], axis=1).dropna(subset=["cancelled_value"]).fillna({"gross": 0}).reset_index()
cust_c["rate"] = cust_c["cancelled_value"] / cust_c["gross"].replace(0, np.nan)
print("\nTop 10 customers by cancelled value:")
print(cust_c.nlargest(10, "cancelled_value").round(2).to_string(index=False))
print(f"\nTop 10 customers' share of all cancelled value: "
      f"{cust_c.nlargest(10, 'cancelled_value')['cancelled_value'].sum() / canc_val:.1%}")

# Days between purchase and cancellation (matched pairs from 4.5b)
fig4 = px.histogram(df_matched[df_matched["Matched"] & (df_matched["DaysToCancel"] <= 90)],
                    x="DaysToCancel", nbins=45,
                    title="Days between purchase and cancellation (matched, up to 90 days)",
                    labels={"DaysToCancel": "Days"})
fig4.show()

print("\nTop 10 products by cancelled value:")
print(prod.nlargest(10, "cancelled_value")[
    ["StockCode", "name", "sold_units", "cancelled_units", "cancelled_value", "return_rate"]
].round(3).to_string(index=False))

print("\nHighest return rates (products with 500+ units sold):")
print(big.nlargest(10, "return_rate")[
    ["StockCode", "name", "sold_units", "cancelled_units", "return_rate"]
].round(3).to_string(index=False))

Cancellation lines: 8,666 (1.6% of lines)
Cancellation invoices: 3,420 vs sales invoices: 19,771 (17.3%)
Cancelled value: 230,248 GBP (2.3% of gross)
Customers with at least one cancellation: 1,532 of 4,361



Top 10 customers by cancelled value:
CustomerID  cancelled_value     gross  rate
     15749         22998.40  44534.30  0.52
     16029         12338.16  72708.09  0.17
     12931          8511.15  42055.96  0.20
     14911          7393.59 136161.83  0.05
     14607          5228.40  15021.50  0.35
     17450          4815.26 194390.79  0.02
     17949          4814.74  58030.48  0.08
     15482          4486.24  10839.76  0.41
     15769          4429.00  56252.72  0.08
     16013          3677.15  37130.60  0.10

Top 10 customers' share of all cancelled value: 34.2%



Top 10 products by cancelled value:
StockCode                               name  sold_units  cancelled_units  cancelled_value  return_rate
    22423           REGENCY CAKESTAND 3 TIER     13851.0            855.0          9697.05        0.062
   85123A WHITE HANGING HEART T-LIGHT HOLDER     37641.0           2578.0          6624.30        0.068
    21108 FAIRY CAKE FLANNEL ASSORTED COLOUR     10241.0           3150.0          6591.42        0.308
    23113              PANTRY CHOPPING BOARD      1154.0            946.0          4803.06        0.820
    48185                 DOORMAT FAIRY CAKE      3615.0            674.0          4554.90        0.186
    21175        GIN + TONIC DIET METAL SIGN     12207.0           2030.0          3775.33        0.166
   47566B             TEA TIME PARTY BUNTING      2379.0           1424.0          3692.95        0.599
    22273               FELTCRAFT DOLL MOLLY      4509.0           1447.0          3512.65        0.321
    71477  COLOUR GLASS. ST

In [36]:
# 1) April spike: largest cancellation lines
apr = canc[canc["YearMonth"] == "2011-04"]
print("April cancelled value:", round(-apr["TotalLineRevenue"].sum(), 0))
print("April gross sales:   ", round(sales_only.loc[sales_only["YearMonth"] == "2011-04", "TotalLineRevenue"].sum(), 0))
print(apr.nsmallest(5, "TotalLineRevenue")[
    ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "TotalLineRevenue", "CustomerID"]
].to_string(index=False))

# 2) Large lines (sales and cancellations) for the two suspicious products
for code in ["84347", "23113"]:
    x = df[(df["StockCode"] == code) & (df["Quantity"].abs() >= 300)]
    print(f"\nStockCode {code} - lines with |Quantity| >= 300:")
    print(x[["InvoiceNo", "Quantity", "UnitPrice", "InvoiceDate", "CustomerID"]]
          .sort_values("InvoiceDate").to_string(index=False))

# 3) Same-customer, same-product, same-quantity reversals of any delay (large only)
s_ = df[(~df["IsCancellation"]) & df["CustomerID"].notna()]
c_ = df[df["IsCancellation"] & df["CustomerID"].notna()].copy()
c_["Quantity"] = -c_["Quantity"]
pp = c_.merge(s_, on=["CustomerID", "StockCode", "Quantity", "UnitPrice"], suffixes=("_c", "_s"))
pp["days"] = (pp["InvoiceDate_c"] - pp["InvoiceDate_s"]).dt.total_seconds() / 86400
pp = pp[pp["days"] >= 0]
print("\nLargest exact reversals at any delay (top 8):")
print(pp.nlargest(8, "Quantity")[
    ["CustomerID", "StockCode", "Quantity", "UnitPrice", "InvoiceNo_s", "InvoiceNo_c", "days"]
].round(2).to_string(index=False))

April cancelled value: 33316.0
April gross sales:    515470.0
InvoiceNo StockCode                        Description  Quantity  UnitPrice  TotalLineRevenue CustomerID
  C550456     21108 FAIRY CAKE FLANNEL ASSORTED COLOUR     -3114       2.10           -6539.4      15749
  C550456    85123A WHITE HANGING HEART T-LIGHT HOLDER     -1930       2.55           -4921.5      15749
  C550456     48185                 DOORMAT FAIRY CAKE      -670       6.75           -4522.5      15749
  C550456     21175        GIN + TONIC DIET METAL SIGN     -2000       1.85           -3700.0      15749
  C550456    47566B             TEA TIME PARTY BUNTING     -1300       2.55           -3315.0      15749

StockCode 84347 - lines with |Quantity| >= 300:
Empty DataFrame
Columns: [InvoiceNo, Quantity, UnitPrice, InvoiceDate, CustomerID]
Index: []

StockCode 23113 - lines with |Quantity| >= 300:
Empty DataFrame
Columns: [InvoiceNo, Quantity, UnitPrice, InvoiceDate, CustomerID]
Index: []

Largest exact reversals

### Insights 6.5 (cancellations)

18. **Cancellations are frequent but small in value.** 3,420 cancellation invoices equal 17.3% of the 19,771 sales invoices, yet they represent only 2.3% of gross sales (~230K GBP). About 35% of customers (1,532 of 4,361) have at least one cancellation.
19. **Cancelled value is concentrated.** The top 10 customers account for 34.2% of cancelled value. Customer 15749 alone returned ~23K GBP (52% of their purchases), about 10% of all cancelled value.
20. **The April 2011 spike (~6.5% cancellation rate) comes from a single invoice.** `C550456` (customer 15749, ~23K GBP) makes up about 70% of April's cancelled value (~33K GBP). It cancels, 97 days later, three lines of invoice `540815` with identical quantities and prices. It is kept as a genuine return (unlike the minutes-long reversals isolated in 4.9).
21. **Cancellations do not rise with the Q4 sales peak.** November, the strongest sales month, has one of the lowest cancellation rates (~1.7%, read from the chart).
22. **High-volume products have normal return rates, while a few products show very high ratios.** `REGENCY CAKESTAND 3 TIER` has the highest cancelled value (~9.7K GBP) at a 6.2% return rate. `ROTATING SILVER ANGELS T-LIGHT HLDR` (99.1%) and `PANTRY CHOPPING BOARD` (82%) have very high ratios without any single large order behind them (no line of 300+ units), so they reflect many small cancellations or purchases before the dataset starts (assumption).
23. **Most returns happen quickly.** The median delay between purchase and cancellation is 9 days, and the majority of matched cancellations occur within the first 10 days.

Findings are descriptive; reasons for returns are not available in the data.

In [37]:
sales_only = df[~df["IsCancellation"]]

inv = sales_only.groupby("InvoiceNo").agg(
    revenue=("TotalLineRevenue", "sum"),
    lines=("StockCode", "size"),
    distinct_products=("StockCode", "nunique"),
    units=("Quantity", "sum"),
    is_uk=("Is_UK", "first"),
    is_guest=("CustomerID", lambda s: s.isna().all()),
    month=("YearMonth", "first"),
).reset_index()

print("Sales invoices:", len(inv))
print(inv[["revenue", "lines", "units"]].describe(percentiles=[.5, .9, .99]).round(1))
print(f"\nMean AOV: {inv['revenue'].mean():,.1f} | Median AOV: {inv['revenue'].median():,.1f}")

# Share of revenue from the top 1% / 10% of invoices
srt = inv["revenue"].sort_values(ascending=False)
print(f"Top 1% of invoices = {srt.head(int(len(srt)*0.01)).sum() / srt.sum():.1%} of revenue")
print(f"Top 10% of invoices = {srt.head(int(len(srt)*0.10)).sum() / srt.sum():.1%} of revenue")

# Segment comparison
print("\nUK vs non-UK:")
print(inv.groupby("is_uk")[["revenue", "lines", "units"]].median().round(1))
print("\nGuest vs registered:")
print(inv.groupby("is_guest")[["revenue", "lines", "units"]].median().round(1))

inv_pos = inv[inv["revenue"] > 0].copy()
inv_pos["log10_revenue"] = np.log10(inv_pos["revenue"])

fig1 = px.histogram(inv_pos, x="log10_revenue", nbins=60,
                    title="Order value distribution (log10 of GBP)",
                    labels={"log10_revenue": "log10(invoice revenue)  [2 = 100 GBP, 3 = 1,000 GBP]"})
fig1.show()

fig2 = px.histogram(inv, x="lines", nbins=60, range_x=[0, 150],
                    title="Basket size: lines per invoice (up to 150)",
                    labels={"lines": "Lines per invoice"})
fig2.show()

fig3 = px.box(inv[inv["revenue"] > 0], x="month", y="revenue", log_y=True,
              title="Order value by month (log scale)",
              labels={"revenue": "Invoice revenue (GBP, log)", "month": "Month"})
fig3.show()

Sales invoices: 19771
       revenue    lines    units
count  19771.0  19771.0  19771.0
mean     505.9     26.4    273.4
std     1178.3     47.3    559.7
min        0.4      1.0      1.0
50%      302.2     15.0    152.0
90%      915.4     53.0    531.0
99%     4393.9    219.0   2229.0
max    52940.9   1113.0  15049.0

Mean AOV: 505.9 | Median AOV: 302.2
Top 1% of invoices = 17.3% of revenue
Top 10% of invoices = 47.6% of revenue

UK vs non-UK:
       revenue  lines  units
is_uk                       
0        385.3   17.0  224.5
1        299.6   15.0  146.0

Guest vs registered:
          revenue  lines  units
is_guest                       
False       301.6   15.0  156.0
True        360.3   38.0   82.0


### Insights 6.6 (order value and baskets)

24. **The typical order is small.** The median invoice is ~302 GBP with 15 lines and 152 units; the mean (~506 GBP) is pulled up by a minority of very large orders. Order values are roughly log-normal, peaking near 300 GBP, with most invoices between ~100 and ~1,000 GBP.
25. **Revenue is concentrated in large orders.** The top 1% of invoices generate 17.3% of revenue and the top 10% generate 47.6%.
26. **Non-UK orders are somewhat larger than UK orders.** Median order value is ~385 GBP outside the UK vs ~300 GBP in the UK (17 vs 15 lines, 225 vs 146 units).
27. **Guest invoices have a different basket shape.** Their median is 38 lines but only 82 units (about 2 units per line), versus 15 lines and 156 units (about 10 per line) for registered customers. The reason cannot be determined from the data.
28. **The Q4 increase comes from order volume, not order size.** The median order value is similar across months (~300 GBP, read from the chart), while invoice counts rise from ~1,080 (Jan) to ~2,751 (Nov).

Findings are descriptive; they do not show why customers order the way they do.

In [38]:
sales_only = df[~df["IsCancellation"]]


def pareto_table(series):
    s = series.sort_values(ascending=False).reset_index(drop=True)
    out = pd.DataFrame({"revenue": s})
    out["rank_pct"] = (out.index + 1) / len(out)
    out["cum_share"] = out["revenue"].cumsum() / out["revenue"].sum()
    return out


def share_at(tbl, pct):
    return tbl.loc[tbl["rank_pct"] <= pct, "cum_share"].iloc[-1]


def items_for(tbl, share):
    n = int((tbl["cum_share"] < share).sum()) + 1
    return n, n / len(tbl)


prod_p = pareto_table(sales_only.groupby("StockCode")["TotalLineRevenue"].sum())
cust_p = pareto_table(
    df_customers[~df_customers["IsCancellation"]].groupby("CustomerID")["TotalLineRevenue"].sum()
)

for name, t in [("Products", prod_p), ("Customers", cust_p)]:
    n80, p80 = items_for(t, 0.80)
    print(f"{name}: {len(t):,} total")
    print(f"  Top 20% generate {share_at(t, 0.20):.1%} of revenue")
    print(f"  Top 10% generate {share_at(t, 0.10):.1%} of revenue")
    print(f"  Top 1%  generate {share_at(t, 0.01):.1%} of revenue")
    print(f"  80% of revenue needs {n80:,} items ({p80:.1%} of all)\n")

fig = go.Figure()
fig.add_trace(go.Scatter(x=prod_p["rank_pct"], y=prod_p["cum_share"], mode="lines", name="Products"))
fig.add_trace(go.Scatter(x=cust_p["rank_pct"], y=cust_p["cum_share"], mode="lines", name="Customers"))
fig.add_hline(y=0.8, line_dash="dash", line_color="gray", annotation_text="80% of revenue")
fig.add_vline(x=0.2, line_dash="dash", line_color="gray", annotation_text="20% of items")
fig.update_layout(
    title="Pareto curve: cumulative revenue share vs share of items",
    xaxis_title="Share of items (ranked by revenue, highest first)",
    yaxis_title="Cumulative share of revenue",
    xaxis_tickformat=".0%", yaxis_tickformat=".0%",
)
fig.show()

Products: 3,912 total
  Top 20% generate 78.3% of revenue
  Top 10% generate 61.1% of revenue
  Top 1%  generate 17.8% of revenue
  80% of revenue needs 841 items (21.5% of all)

Customers: 4,333 total
  Top 20% generate 73.9% of revenue
  Top 10% generate 60.3% of revenue
  Top 1%  generate 30.5% of revenue
  80% of revenue needs 1,163 items (26.8% of all)



### Insights 6.7 (concentration)

29. **Product revenue roughly follows the 80/20 rule.** The top 20% of products (of 3,912) generate 78.3% of gross revenue, and 841 products (21.5%) are needed to reach 80%.
30. **Customer revenue is somewhat less concentrated overall, but more concentrated at the very top.** The top 20% of customers (of 4,333 with sales) generate 73.9% of revenue, and 26.8% are needed to reach 80%. However, the top 1% of customers alone generate 30.5% of revenue, versus 17.8% for the top 1% of products.

Rankings use gross sales revenue (cancellations excluded) and registered customers only; guest checkouts are not included in the customer curve.

In [39]:
cutoff = df["InvoiceDate"].max().normalize() - pd.Timedelta(days=90) + pd.Timedelta(days=1)
print("Last 90 days start:", cutoff.date(), "| data ends:", df["InvoiceDate"].max().date())

before = df[df["InvoiceDate"] < cutoff]
last90 = df[df["InvoiceDate"] >= cutoff]

rev_before = before.groupby("StockCode")["TotalLineRevenue"].sum()
rev_last = last90.groupby("StockCode")["TotalLineRevenue"].sum()

hist = pd.DataFrame({"rev_before": rev_before}).join(rev_last.rename("rev_last90")).fillna({"rev_last90": 0})
hist = hist[hist["rev_before"] > 0]

threshold = hist["rev_before"].quantile(0.80)
strong = hist[hist["rev_before"] >= threshold].copy()

last_sale = (df[~df["IsCancellation"]].groupby("StockCode")["InvoiceDate"].max().rename("last_sale"))
names = (df[~df["IsCancellation"]].groupby("StockCode")["Description"]
         .agg(lambda s: s.mode().iloc[0] if s.notna().any() else None).rename("name"))
strong = strong.join(last_sale).join(names)

dead = strong[strong["rev_last90"] <= 0].sort_values("rev_before", ascending=False).reset_index()
dead["days_since_last_sale"] = (df["InvoiceDate"].max() - dead["last_sale"]).dt.days

print(f"Products with revenue before the cutoff: {len(hist):,}")
print(f"'Historically strong' (top 20%, >= {threshold:,.0f} GBP): {len(strong):,}")
print(f"Of those, zero/negative revenue in last 90 days: {len(dead):,} ({len(dead) / len(strong):.1%})")
print(f"Their historical revenue: {dead['rev_before'].sum():,.0f} GBP "
      f"({dead['rev_before'].sum() / strong['rev_before'].sum():.1%} of the strong group)")

print("\nTop 15 dead-stock candidates:")
print(dead.head(15)[["StockCode", "name", "rev_before", "rev_last90", "last_sale", "days_since_last_sale"]]
      .round(0).to_string(index=False))

top_dead = dead.head(15).sort_values("rev_before")
fig = px.bar(top_dead, x="rev_before", y="name", orientation="h",
             title="Top 15 dead-stock candidates by historical revenue (GBP)",
             labels={"rev_before": "Net revenue before last 90 days (GBP)", "name": ""})
fig.show()

# When did these products last sell? (monthly count of candidates by last-sale month)
dead["last_sale_month"] = dead["last_sale"].dt.to_period("M").astype(str)
fig2 = px.histogram(dead.sort_values("last_sale_month"), x="last_sale_month",
                    title="Dead-stock candidates by month of last sale",
                    labels={"last_sale_month": "Month of last sale"})
fig2.show()

Last 90 days start: 2011-09-11 | data ends: 2011-12-09
Products with revenue before the cutoff: 3,659
'Historically strong' (top 20%, >= 1,901 GBP): 732
Of those, zero/negative revenue in last 90 days: 16 (2.2%)
Their historical revenue: 57,811 GBP (1.2% of the strong group)

Top 15 dead-stock candidates:
StockCode                                name  rev_before  rev_last90           last_sale  days_since_last_sale
   47556B                TEA TIME TEA TOWELS       6045.0         0.0 2011-04-18 13:20:00                   234
    22783    SET 3 WICKER OVAL BASKETS W LIDS      5961.0         0.0 2011-06-14 11:28:00                   178
    22088         PAPER BUNTING COLOURED LACE      5512.0         0.0 2011-07-06 10:41:00                   156
    21033     JUMBO BAG CHARLIE AND LOLA TOYS      4850.0         0.0 2011-09-08 13:53:00                    91
    21658                  GLASS  BEURRE DISH      3596.0       -11.0 2011-08-11 10:14:00                   120
    22926      IVORY 

### Insights 6.8 (dead stock)

31. **Dead stock is limited.** Of 732 historically strong products (top 20% by net revenue before 2011-09-11, >= 1,901 GBP), only 16 (2.2%) have zero or negative net revenue in the last 90 days. Their historical revenue is 57,811 GBP, 1.2% of the strong group.
32. **Candidates stopped selling gradually.** Last sales are spread from January to early September 2011 (peak in Jul-Aug, 6 products), with no single stopping date. The oldest are `UTILTY CABINET WITH HOOKS` (326 days) and `HALL CABINET WITH 3 DRAWERS` (309 days). Several candidates last sold within ~10 days of the cutoff (e.g. `JUMBO BAG CHARLIE AND LOLA TOYS`, 91 days), so they may not be dead.

The candidates are for review, not confirmed dead stock: seasonal items or deliberately discontinued products cannot be distinguished from one year of data.

In [40]:
cs = df_customers[~df_customers["IsCancellation"]]

per_inv = cs.groupby(["CustomerID", "InvoiceNo"]).agg(
    revenue=("TotalLineRevenue", "sum"),
    units=("Quantity", "sum"),
    date=("InvoiceDate", "min"),
).reset_index()

cb = per_inv.groupby("CustomerID").agg(
    orders=("InvoiceNo", "nunique"),
    gross=("revenue", "sum"),
    avg_order=("revenue", "mean"),
    avg_units=("units", "mean"),
    first=("date", "min"),
    last=("date", "max"),
)
cb["active_days"] = (cb["last"] - cb["first"]).dt.days

print("Customers with sales:", len(cb))
print(cb[["orders", "gross", "avg_order", "avg_units"]].describe(percentiles=[.5, .9, .99]).round(1))
print(f"\nOne-time buyers (1 order): {(cb['orders'] == 1).sum():,} ({(cb['orders'] == 1).mean():.1%})")
print(f"Repeat buyers (2+ orders): {(cb['orders'] >= 2).sum():,} ({(cb['orders'] >= 2).mean():.1%})")

# Days between consecutive orders (repeat buyers)
gaps = (per_inv.sort_values(["CustomerID", "date"])
        .groupby("CustomerID")["date"].diff().dt.days.dropna())
print(f"Median days between consecutive orders: {gaps.median():.0f}")

# Wholesale-like indicator
thr = cb["avg_units"].quantile(0.90)
cb["wholesale_like"] = cb["avg_units"] >= thr
w = cb.groupby("wholesale_like").agg(
    customers=("orders", "size"),
    median_orders=("orders", "median"),
    median_avg_order=("avg_order", "median"),
    median_avg_units=("avg_units", "median"),
    revenue=("gross", "sum"),
)
w["revenue_share"] = w["revenue"] / w["revenue"].sum()
print(f"\nThreshold (top 10% avg units per order): {thr:,.0f} units")
print(w.round(2).to_string())

country = df_customers.groupby("CustomerID")["Country"].agg(lambda s: s.mode().iloc[0])
cb = cb.join(country)
print("\nShare of wholesale-like customers that are non-UK:",
      f"{(cb.loc[cb['wholesale_like'], 'Country'] != 'United Kingdom').mean():.1%}",
      "| non-UK share among all customers:",
      f"{(cb['Country'] != 'United Kingdom').mean():.1%}")

fig1 = px.histogram(cb, x="orders", nbins=50, range_x=[0, 40],
                    title="Orders per customer (up to 40)",
                    labels={"orders": "Orders per customer"})
fig1.show()

fig2 = px.scatter(cb.reset_index(), x="orders", y="avg_order", color="wholesale_like",
                  log_x=True, log_y=True, hover_name="CustomerID",
                  title="Orders vs average order value (log-log), wholesale-like highlighted",
                  labels={"orders": "Orders (log)", "avg_order": "Avg order value GBP (log)"})
fig2.show()

fig3 = px.histogram(gaps[gaps <= 120], nbins=40,
                    title="Days between consecutive orders (repeat buyers, up to 120 days)",
                    labels={"value": "Days between orders"})
fig3.show()

Customers with sales: 4333
       orders     gross  avg_order  avg_units
count  4333.0    4333.0     4333.0     4333.0
mean      4.2    1959.7      378.3      228.9
std       7.6    8461.2      510.2      309.9
min       1.0       2.9        2.9        1.0
50%       2.0     661.3      289.5      161.0
90%       9.0    3556.4      657.1      420.6
99%      30.0   17569.5     2005.8     1342.3
max     206.0  279138.0    14844.8     7824.0

One-time buyers (1 order): 1,505 (34.7%)
Repeat buyers (2+ orders): 2,828 (65.3%)
Median days between consecutive orders: 22

Threshold (top 10% avg units per order): 421 units
                customers  median_orders  median_avg_order  median_avg_units     revenue  revenue_share
wholesale_like                                                                                         
False                3899            2.0            261.18             144.0  5299203.13           0.62
True                  434            2.0            848.99           

### Insights 6.9 (customer behavior and wholesale indicators)


33. **Typical customer is a small-frequency, mid-size buyer.** Among 4,333 customers with sales, the median is 2 orders and 661 GBP gross spend, while the mean is 4.2 orders and 1,960 GBP. A small number of heavy customers pulls the mean up (max 206 orders, 279,138 GBP).
34. **About one third never returned.** 1,505 customers (34.7%) placed a single order; 2,828 (65.3%) placed two or more.
35. **Repeat buyers order roughly every three weeks.** The median gap between consecutive orders is 22 days, with many gaps under 5 days.
36. **Order sizes are large.** The median customer averages 161 units and 289 GBP per order, which is more consistent with trade-style buying than with individual shoppers. This is an indication only: the data has no customer-type field.
37. **Large-order customers carry a disproportionate share of revenue.** The 434 customers in the top 10% by average units per order (>= 421 units) generate 38% of gross revenue.
38. **Large orders are not the same as frequent orders.** Wholesale-like customers have the same median number of orders (2) as the rest, so the indicator captures order size, not repeat behavior. Many are one-time large buyers.
39. **International customers are over-represented among wholesale-like customers.** 22.8% are outside the UK versus 9.6% of all customers. This is descriptive and does not explain why.

The wholesale-like flag is a definition we chose (top 10% average units per order), not a confirmed classification.

## 7. Business Insights

The 12 findings below are selected from Section 6 (numbers in brackets refer to the insight numbering there). Each is linked to a business action. Actions are suggestions based on descriptive data, not proven causal effects.

### Time and seasonality

| # | Insight | Evidence | Suggested action |
|---|---|---|---|
| 1 | Revenue roughly doubles in Q4 | Oct-Nov averages ~1.24M GBP/month vs ~634K for Jan-Aug 2011 (1.96x) [15] | Plan inventory and marketing capacity for Q4 |
| 2 | The ramp-up starts in September | Average daily revenue rises from ~26-27K (Jul-Aug) to ~38.9K (Sep) [16] | Start stock build-up and campaigns in August/September, not October |
| 3 | Q4 growth comes from more orders, not bigger orders | Median order value stays ~300 GBP; invoices rise from ~1,080 (Jan) to ~2,751 (Nov) [28] | Focus Q4 on traffic and reactivation rather than basket-size upselling |
| 4 | Sales are concentrated on weekdays and daytime | Thursday and Tuesday are strongest; most sales fall between 09:00 and 16:00; no Saturday transactions recorded [9, 10, 11] | Time email campaigns and ad spend for the daytime peak; schedule maintenance outside it. Saturday gap is unexplained, so treat it as missing data |

### Customers and concentration

| # | Insight | Evidence | Suggested action |
|---|---|---|---|
| 5 | Revenue depends on large orders | Top 10% of invoices generate 47.6% of revenue [25] | Offer dedicated service and bulk pricing for large-order accounts |
| 6 | A small group of customers carries a large share | Top 1% of customers generate 30.5% of revenue; top 20% generate 73.9% [30] | Prioritize retention for the top tier (loyalty perks, early access) |
| 7 | Large-order customers are not necessarily frequent | 434 "wholesale-like" customers (>= 421 units/order) generate 38% of gross revenue, yet their median is 2 orders, same as the rest [37, 38] | Test trade-account offers and repeat-purchase incentives for large one-time buyers |
| 8 | About one third of customers never returned | 1,505 of 4,333 customers (34.7%) placed a single order [34] | Run a welcome sequence aimed at a second purchase |
| 9 | Repeat buyers return about every three weeks | Median gap between orders is 22 days [35] | Use the 22-day rhythm as a reference for win-back triggers; validate the threshold in RFM |

### Products, returns, and geography

| # | Insight | Evidence | Suggested action |
|---|---|---|---|
| 10 | Product revenue follows roughly an 80/20 pattern; dead stock is limited | Top 20% of products generate 78.3% of gross revenue [29]; only 16 of 732 strong products (2.2%) stopped selling in the last 90 days [31] | Protect availability of top products; review the 16 candidates individually instead of running broad clearance |
| 11 | Cancellations are frequent but small and concentrated | 17.3% of sales invoices but only 2.3% of gross sales; top 10 customers account for 34.2% of cancelled value [18, 19] | Monitor returns per customer and review top returners rather than changing policy globally |
| 12 | Non-UK revenue is fragile in some countries and broad in others | UK is 84.8% of net revenue; Netherlands, EIRE, Australia rely on 3-9 customers; Germany and France have 95 and 87 customers with AOV close to the UK [12, 13, 14] | Manage the few-account countries as key accounts; treat Germany and France as markets to test for growth |

### Overall reading

The business is highly seasonal and concentrated: a large base of low-frequency customers, with a small group of large buyers carrying a disproportionate share of revenue. This supports building RFM and segments so that retention, win-back, and onboarding actions can be targeted instead of applied to everyone equally.

*All findings are descriptive, based on one year of data, and registered customers only where customer-level.*

In [42]:
import os
os.makedirs("../data", exist_ok=True)

df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)
df["CustomerID"] = df["CustomerID"].astype("string")

df.to_parquet("../data/clean_transactions.parquet", index=False)

check = pd.read_parquet("../data/clean_transactions.parquet")
print("Saved rows:", len(check))
print("Matches df:", len(check) == len(df))
print("Customers:", check["CustomerID"].nunique())
print("Net revenue (GBP):", round(check["TotalLineRevenue"].sum(), 2))
print(check.dtypes)

Saved rows: 531235
Matches df: True
Customers: 4361
Net revenue (GBP): 9772003.97
InvoiceNo                   object
StockCode                   object
Description                 object
Quantity                     int64
InvoiceDate         datetime64[ns]
UnitPrice                  float64
CustomerID          string[python]
Country                     object
Year                         int32
Month                        int32
DayOfWeek                   object
Hour                         int32
IsCancellation                bool
TotalLineRevenue           float64
Is_UK                        int64
YearMonth                   object
dtype: object
